# Chapter 1: Keras / TensorFlow — The Easiest On-Ramp

### Why This Matters

Keras (now TensorFlow's official high-level API) is deliberately designed to be the most approachable entry point into deep learning — its `Sequential` API reads almost like a list of layers, and sensible defaults mean you can get a working model training in a handful of lines.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# A simple binary classification network — e.g., for the churn dataset from Volumes 05-07
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(n_features,)),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    verbose=1
)

pd.DataFrame(history.history)[["loss", "val_loss", "auc", "val_auc"]].plot()

Walking through each piece, since this exact `Sequential → compile → fit` shape reappears everywhere in Keras code you'll encounter:

- **`Dense(64, activation="relu")`** — a "fully connected" layer with 64 neurons, each connected to every input. `relu` (Rectified Linear Unit — outputs the input directly if positive, else zero) is the standard default activation for hidden layers, chosen largely because it trains faster and more reliably than older alternatives (like sigmoid) in deep networks.
- **`Dropout(0.3)`** — during training only, randomly "turns off" 30% of the previous layer's neurons on each batch. This is a regularization technique specifically to fight overfitting, forcing the network to not over-rely on any single neuron or narrow combination of them — conceptually similar in purpose to the `subsample`/`colsample_bytree` randomness in Volume 06's gradient boosting.
- **The final `Dense(1, activation="sigmoid")`** — one output neuron, squashed to a 0-1 probability by the sigmoid function, exactly matching the shape of a binary classification target. For multi-class problems, this becomes `Dense(n_classes, activation="softmax")` instead.
- **`compile(optimizer="adam", loss="binary_crossentropy", ...)`** — `adam` is the near-universal default optimizer (an adaptive variant of gradient descent that works well with minimal tuning across most problems); the `loss` function must match your problem type (`binary_crossentropy` for binary classification, `categorical_crossentropy` for multi-class, `mse` for regression).
- **`epochs=30, batch_size=64`** — an **epoch** is one complete pass through the entire training dataset; a **batch** is a small chunk of data processed together before one weight update. Training on mini-batches rather than the whole dataset at once (or one row at a time) is a deliberate, standard compromise between computational efficiency and training stability.
- **`validation_split=0.2`** — automatically holds out 20% of the *training* data (separate from your already-held-out test set) to monitor for overfitting during training itself, which is exactly what the loss-curve plot at the end visualizes: if `val_loss` starts rising while `loss` keeps falling, that's the network beginning to overfit, and it's your cue to stop training earlier (see `EarlyStopping` below) or add more regularization.

### 1.1 The Functional API — For Anything Beyond a Simple Stack

In [ ]:
inputs_numeric = keras.Input(shape=(8,))
inputs_categorical = keras.Input(shape=(4,))
merged = layers.Concatenate()([inputs_numeric, inputs_categorical])
output = layers.Dense(1, activation="sigmoid")(merged)
model = keras.Model([inputs_numeric, inputs_categorical], output)

`Sequential` only handles a simple, single-path stack of layers. The moment your architecture needs **multiple inputs** (e.g., numeric features and a separate embedding path for a text field), **multiple outputs**, or **shared layers** reused in more than one place, you need the **Functional API** — where you build the graph by explicitly calling each layer on the output of the previous one, giving you full control over the network's shape.

### 1.2 Callbacks — Automating Good Training Practice

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model.keras", save_best_only=True)

model.fit(X_train, y_train, validation_split=0.2, epochs=100,
          callbacks=[early_stop, checkpoint])

`EarlyStopping` directly automates the "stop before overfitting" instinct from the loss-curve discussion above: it monitors `val_loss` and halts training if it hasn't improved for `patience` epochs, restoring the weights from the best epoch rather than the final (possibly overfit) one. This is a near-default habit worth adopting in essentially every neural network you train, since guessing the "right" number of epochs upfront is genuinely difficult and this makes the guess unnecessary.

### Cheat Sheet — Keras/TensorFlow

| Task | Code |
|---|---|
| Simple stack | `keras.Sequential([layers.Dense(...), ...])` |
| Multi-input/branching | Functional API: `keras.Model(inputs, outputs)` |
| Compile | `model.compile(optimizer="adam", loss=..., metrics=[...])` |
| Train | `model.fit(X, y, epochs=, batch_size=, validation_split=)` |
| Stop overfitting automatically | `EarlyStopping(monitor="val_loss", patience=5)` |
| Save best model | `ModelCheckpoint(path, save_best_only=True)` |
| Save / load | `model.save(path)`, `keras.models.load_model(path)` |

---